In [ ]:
from pathlib import Path
import pandas as pd

relative_paths = (
    Path("imdb_movies.csv"),
    Path("FinetuningofQ&AClassification/imdb_movies.csv"),
    Path("Training-model/FinetuningofQ&AClassification/imdb_movies.csv"),
)
csv_path = next(
    (base / relative for base in (Path.cwd(), *Path.cwd().parents)
     for relative in relative_paths if (base / relative).is_file()),
    None,
)

if csv_path is None:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            f"Place imdb_movies.csv in {Path.cwd()} and rerun this cell."
        ) from None

    print("Select imdb_movies.csv from your computer in the upload picker.")
    uploaded = files.upload()
    if "imdb_movies.csv" not in uploaded:
        raise FileNotFoundError("Upload imdb_movies.csv, then rerun this cell.")
    # Save the returned bytes explicitly, including when Colab renames a file.
    csv_path = Path.cwd() / "imdb_movies.csv"
    csv_path.write_bytes(uploaded["imdb_movies.csv"])
    del uploaded

df = pd.read_csv(csv_path)
print(f"Loaded {len(df):,} rows from {csv_path}")

Select imdb_movies.csv from your computer in the upload picker.


KeyboardInterrupt: 

In [ ]:
df

## Build the Context For Questions

In [ ]:
def build_context_string(row):
    context = f"{row['names']} is a {row['genre']} movie released in {row['date_x']} with budget of {row['budget_x']} had revenue {row['revenue']}."
    
    if(description := row['overview']):
        context += f"The movie is all about {description}"
        
    return context

### Helper function for Q&A Pairs

In [ ]:
def build_question_string(row, context):
    title = str(row['names'])
    genre = str(row['genre'])
    budget = str(row['budget_x'])
    revenue = str(row['revenue'])
    year = str(row['date_x'])
    description = str(row['overview'])
    
    qa_list = []
    
    #Example question: Budget
    question_1 = f"What is the budget of the movie {title}?"
    answer_1 = budget
    answer_start_1 = context.find(answer_1)
    if answer_start_1 != -1:
        qa_list.append({
            'question': question_1,
            'id': f"{title}_budget",
            'answers': [
                {
                    'text': answer_1,
                    'answer_start': answer_start_1
                }
            ],
            'is_impossible': False
        })
    
    #Example question: Revenue
    question_2 = f"What is the revenue of the movie {title}?"
    answer_2 = revenue
    answer_start_2 = context.find(answer_2)
    if answer_start_2 != -1:
        qa_list.append({
            'question': question_2,
            'id': f"{title}_revenue",
            'answers': [
                {
                    'text': answer_2,
                    'answer_start': answer_start_2
                }
            ],
            'is_impossible': False
        })
        
    #Example question: Genre
    question_3 = f"What is the genre of the movie {title}?"
    answer_3 = genre
    answer_start_3 = context.find(answer_3)
    if answer_start_3 != -1:
        qa_list.append({
            'question': question_3,
            'id': f"{title}_genre",
            'answers': [
                {
                    'text': answer_3,
                    'answer_start': answer_start_3
                }
            ],
            'is_impossible': False
        })
    
    #Example question: Year
    question_4 = f"What is the year of release of the movie {title}?"
    answer_4 = year
    answer_start_4 = context.find(answer_4)
    if answer_start_4 != -1:
        qa_list.append({
            'question': question_4,
            'id': f"{title}_year",
            'answers': [
                {
                    'text': answer_4,
                    'answer_start': answer_start_4
                }
            ],
            'is_impossible': False
        })
    
    question_5 = f"Describe the movie {title}."
    answer_5 = description
    answer_start_5 = context.find(answer_5)
    if answer_start_5 != -1:
        qa_list.append({
            'question': question_5,
            'id': f"{title}_description",
            'answers': [
                {
                    'text': answer_5,
                    'answer_start': answer_start_5
                }
            ],
            'is_impossible': False
        })
        
  
    return qa_list

### Build SQuaD style JSON
We need SQuAD-like formatting if we intend to train an extractive question-answering model using the Hugging Face QA pipeline (e.g., DistilBertForQuestionAnswering fine-tuned in the same way as on SQuAD). 

In other words, the SQuAD-like structure is **required** by the standard HuggingFace QA training scripts 

In [ ]:
def build_squad_dataset(df):
  data_list = []
      
  for idx, row in df.iterrows():
      context = build_context_string(row)
      
      qa_list = build_question_string(row, context)
      if(len(qa_list) == 0):
          continue
      
      paragraph_data = {
          'context': context,
          'qas': qa_list
      }
      
      data_list.append({
          'title': row['names'],
          'paragraphs': [paragraph_data]
      })
      
  squad_dict = {
    'title': 'IMDB Movies QA V1',
    'data': data_list
  }

  return squad_dict     
        

In [ ]:
squad_data = build_squad_dataset(df)
squad_data

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict

def squad_dict_to_dataset(squad_dict):
    
    records = []
    
    for entry in squad_dict["data"]:
        title = entry['title']
        for paragraph in entry['paragraphs']:
            context = paragraph['context']
            qas = paragraph['qas']
            for qa in qas:
                question = qa['question']
                id = qa['id']
                is_impossible = qa['is_impossible']
                answers = qa['answers']
                answer_starts = [answer['answer_start'] for answer in answers]
                answer_texts = [answer['text'] for answer in answers]
                
                records.append({
                    'title': title,
                    'context': context,
                    'question': question,
                    'id': id,
                    'is_impossible': is_impossible,
                    'answers': answers,
                    'answer_starts': answer_starts,
                    'answer_texts': answer_texts
                })
    
    return Dataset.from_pandas(pd.DataFrame(records))
            

In [ ]:
dataset = squad_dict_to_dataset(squad_data)
dataset

### Test/Train/Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

# 70% train, 30% test
train_test = dataset.train_test_split(test_size=0.3)

train_dataset = train_test['train']
test_dataset = train_test['test']

# 20% test, 10% validation
validation_dataset = test_dataset.train_test_split(test_size=1/3)

test_dataset = validation_dataset['train']
validation_dataset = validation_dataset['test']

train_dataset, test_dataset, validation_dataset

### Define tokenizer 

In [ ]:
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForQuestionAnswering,
    Trainer,
    TrainingArguments,
    default_data_collator
)

import torch

checkpoint_model = "distilbert-base-uncased"

tokenizer = DistilBertTokenizerFast.from_pretrained(checkpoint_model)

In [ ]:
max_length = 384  # Max length of the encoding
doc_stride = 128  # Stride to handle long contexts

def prepare_train_features(examples):
    # 1) Tokenize questions + contexts
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        max_length=max_length,
        truncation="only_second",    # We only truncate the context if it's too long
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    
#stride=doc_stride – number of overlapping tokens between chunks when a long context is split into multiple windows.
# return_overflowing_tokens=True – return extra chunks/windows when the context is too long, instead of just dropping the overflow.
# return_offsets_mapping=True – return character start/end offsets for each token, so tokens can be mapped back to the original text.
# padding="max_length" – pad every encoded example to exactly max_length tokens.



    # 2) "overflow_to_sample_mapping" indicates which original sample each chunk belongs to
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    print("sample_mapping", sample_mapping)
    # 3) "offset_mapping" provides (char_start, char_end) for each token in the original text
    offset_mapping = tokenized.pop("offset_mapping")
    print("offset_mapping", offset_mapping)
    # 4) We'll need the answer information to map them back to token indices
    answers = examples["answers"]
    
    start_positions = []
    end_positions = []
    
    # 5) Loop over each tokenized chunk
    for i, offsets in enumerate(offset_mapping):
        # The current chunk corresponds to the original example index:
        sample_idx = sample_mapping[i]
        
        # Each example can have 1 or more answers; here it's typically 1
        answer = answers[sample_idx]
        
        start_char = answer[0]["answer_start"]
        end_char = start_char + len(answer[0]["text"])
        
        sequence_ids = tokenized.sequence_ids(i)
        
        ctx_start = 0
        while sequence_ids[ctx_start] != 1:
            ctx_start += 1
        ctx_end = len(tokenized["input_ids"][i]) - 1
        while sequence_ids[ctx_end] != 1:
            ctx_end -= 1
        
        # If the answer text is not in this chunk (overflow window),
        # set start_positions/end_positions to something neutral (e.g. the start of the context)
        if not (start_char < offsets[ctx_end][1] and end_char > offsets[ctx_start][0]):
            start_positions.append(ctx_start)
            end_positions.append(ctx_start)
            continue
        
        # Otherwise, find the first token that starts after or at the answer’s start_char
        start_idx = ctx_start
        while start_idx <= ctx_end and offsets[start_idx][0] <= start_char:
            start_idx += 1
        start_positions.append(start_idx - 1)
        
        # Similarly, find the last token that ends before or at the answer’s end_char
        end_idx = ctx_end
        while end_idx >= ctx_start and offsets[end_idx][1] >= end_char:
            end_idx -= 1
        end_positions.append(end_idx + 1)
    
    # 6) Store these positions in the returned dictionary
    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

### Create DataSet


In [ ]:
dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset,
    'validation': validation_dataset
})

dataset

NameError: name 'DatasetDict' is not defined

In [ ]:
dataset['train'].column_names

In [ ]:
train_dataset = train_dataset.map(prepare_train_features, batched=True, remove_columns=dataset["train"].column_names)
validation_dataset = validation_dataset.map(prepare_train_features, batched=True, remove_columns=dataset["validation"].column_names)

train_dataset, validation_dataset

In [ ]:
model = DistilBertForQuestionAnswering.from_pretrained(checkpoint_model)

training_args = TrainingArguments(
    output_dir= "./distilbert-imdb-qa",
    evaluation_strategy="epoch",
    learning_rate= 3e-5,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=2,
    save_total_limit=1,
    save_steps=500,
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    tokenizer=tokenizer,
    data_collator=default_data_collator
)

trainer.train()

In [ ]:
trainer.save_model("./distilbert-imdb-qa-finetuned")

## Inference with Pipeline code of HF

In [ ]:
df.iloc[0]
context = build_context_string(df.iloc[0])
context

In [ ]:
from transformers import pipeline

qa_pipeline = pipeline("question-answering", model="./distilbert-imdb-qa-finetuned")

data_row = df.iloc[0]
context = build_context_string(data_row)
question = f"Whats the budget of the movie {data_row['names']}?"

print("\n Context: \n", context)
print("\n Question: \n", question)
print("\n Answer: \n", qa_pipeline(question=question, context=context))